# 03 — Predictor source inventory and temporal audit

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


# AKI V2 — Predictor/Feature Inventory

Bu notebook yalnızca toplulaştırılmış veri-kapsam ve kalite denetimleri üretir. Model eğitmez.

In [ ]:
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = "physionet-data.eicu_crd"
WORK_DATASET_NAME = globals().get("WORK_DATASET_NAME") or os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
BQ_LOCATION = "US"
DRIVE_OUTPUT_DIR = f"{OUTPUT_ROOT}/03_FEATURE_INVENTORY_OUTPUTS"
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

In [ ]:
!pip -q install google-cloud-bigquery pandas pyarrow db-dtypes openpyxl

from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd
import os, json, zipfile, hashlib, datetime

auth.authenticate_user()
drive.mount('/content/drive')
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print('BigQuery client ready:', PROJECT_ID)
print('Target dataset:', TARGET_DATASET)
print('Output directory:', DRIVE_OUTPUT_DIR)

In [ ]:
def run_aggregate(label, sql, filename):
    print('Running:', label)
    df = client.query(sql).to_dataframe()
    display(df.head(100))
    path = os.path.join(DRIVE_OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print('Saved:', path, 'rows=', len(df))
    return df


## 01_cohort_integrity_audit.sql

In [ ]:
SQL_01 = f"""
-- Shareable aggregate integrity audit of the locked main cohort.
WITH main AS (
  SELECT *
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
)
SELECT
  COUNT(*) AS cohort_rows,
  COUNT(DISTINCT patientUnitStayID) AS distinct_unit_stays,
  COUNT(DISTINCT uniquePID) AS distinct_patients,
  COUNT(DISTINCT hospitalID) AS hospitals,
  COUNTIF(outcome_creatinine_stage23 = 1) AS events,
  COUNTIF(outcome_creatinine_stage23 = 0) AS nonevents,
  SAFE_DIVIDE(COUNTIF(outcome_creatinine_stage23 = 1), COUNT(*)) AS event_rate,
  COUNT(*) - COUNT(DISTINCT uniquePID) AS duplicate_patient_rows,
  COUNTIF(reference_creatinine IS NULL) AS missing_reference_creatinine,
  COUNTIF(max_stage_by_12h >= 2) AS stage23_present_at_landmark,
  COUNTIF(strict_chronic_dialysis_esrd = 1) AS strict_chronic_dialysis_rows,
  COUNTIF(deterministic_eligible_patient_stay_rank != 1) AS non_rank1_rows
FROM main;
"""
result_01 = run_aggregate(
    "01_cohort_integrity_audit.sql",
    SQL_01,
    "01_cohort_integrity_audit.csv"
)

## 02_source_column_inventory.sql

In [ ]:
SQL_02 = f"""
-- Shareable source-schema inventory for candidate predictor tables.
SELECT
  table_name,
  ordinal_position,
  column_name,
  data_type
FROM `{SOURCE_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN (
  'patient','hospital','lab','vitalperiodic','vitalaperiodic',
  'pasthistory','treatment','infusiondrug','respiratorycare',
  'respiratorycharting','nursecharting'
)
ORDER BY table_name, ordinal_position;
"""
result_02 = run_aggregate(
    "02_source_column_inventory.sql",
    SQL_02,
    "02_source_column_inventory.csv"
)

## 03_demographic_coverage_audit.sql

In [ ]:
SQL_03 = f"""
-- Shareable coverage and plausibility audit for static predictors.
WITH main AS (
  SELECT patientUnitStayID, hospitalID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
d AS (
  SELECT
    m.patientUnitStayID,
    CASE WHEN p.age = '> 89' THEN 90 ELSE SAFE_CAST(p.age AS INT64) END AS age_num,
    LOWER(TRIM(p.gender)) AS gender,
    NULLIF(TRIM(p.ethnicity), '') AS ethnicity,
    SAFE_CAST(p.admissionHeight AS FLOAT64) AS height_cm,
    SAFE_CAST(p.admissionWeight AS FLOAT64) AS weight_kg,
    p.unitType,
    p.unitAdmitSource,
    p.hospitalAdmitSource,
    h.region,
    h.numBedsCategory,
    h.teachingStatus
  FROM main m
  JOIN `{SOURCE_DATASET}.patient` p USING (patientUnitStayID)
  LEFT JOIN `{SOURCE_DATASET}.hospital` h USING (hospitalID)
)
SELECT metric, value
FROM (
  SELECT 'main_cohort' AS metric, COUNT(*) AS value FROM d UNION ALL
  SELECT 'age_available', COUNTIF(age_num IS NOT NULL) FROM d UNION ALL
  SELECT 'female', COUNTIF(gender = 'female') FROM d UNION ALL
  SELECT 'male', COUNTIF(gender = 'male') FROM d UNION ALL
  SELECT 'gender_other_or_missing', COUNTIF(gender IS NULL OR gender NOT IN ('female','male')) FROM d UNION ALL
  SELECT 'ethnicity_available', COUNTIF(ethnicity IS NOT NULL) FROM d UNION ALL
  SELECT 'height_available', COUNTIF(height_cm IS NOT NULL) FROM d UNION ALL
  SELECT 'height_plausible_100_250_cm', COUNTIF(height_cm BETWEEN 100 AND 250) FROM d UNION ALL
  SELECT 'weight_available', COUNTIF(weight_kg IS NOT NULL) FROM d UNION ALL
  SELECT 'weight_plausible_25_300_kg', COUNTIF(weight_kg BETWEEN 25 AND 300) FROM d UNION ALL
  SELECT 'bmi_plausible_10_80', COUNTIF(height_cm BETWEEN 100 AND 250 AND weight_kg BETWEEN 25 AND 300
      AND SAFE_DIVIDE(weight_kg, POW(height_cm/100.0,2)) BETWEEN 10 AND 80) FROM d UNION ALL
  SELECT 'unit_type_available', COUNTIF(NULLIF(TRIM(unitType),'') IS NOT NULL) FROM d UNION ALL
  SELECT 'unit_admit_source_available', COUNTIF(NULLIF(TRIM(unitAdmitSource),'') IS NOT NULL) FROM d UNION ALL
  SELECT 'hospital_admit_source_available', COUNTIF(NULLIF(TRIM(hospitalAdmitSource),'') IS NOT NULL) FROM d UNION ALL
  SELECT 'hospital_region_available', COUNTIF(NULLIF(TRIM(region),'') IS NOT NULL) FROM d UNION ALL
  SELECT 'hospital_bed_category_available', COUNTIF(NULLIF(TRIM(numBedsCategory),'') IS NOT NULL) FROM d UNION ALL
  SELECT 'hospital_teaching_status_available', COUNTIF(teachingStatus IS NOT NULL) FROM d
)
ORDER BY metric;
"""
result_03 = run_aggregate(
    "03_demographic_coverage_audit.sql",
    SQL_03,
    "03_demographic_coverage_audit.csv"
)

## 04_lab_name_coverage_audit.sql

In [ ]:
SQL_04 = f"""
-- Shareable first-12-hour laboratory-name, unit and distribution inventory.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
x AS (
  SELECT
    LOWER(TRIM(l.labName)) AS lab_name,
    COALESCE(NULLIF(TRIM(l.labMeasureNameSystem),''),'[missing]') AS unit_system,
    SAFE_CAST(l.labResult AS FLOAT64) AS value,
    l.patientUnitStayID
  FROM main m
  JOIN `{SOURCE_DATASET}.lab` l USING (patientUnitStayID)
  WHERE l.labResultOffset BETWEEN 0 AND 720
    AND LOWER(TRIM(l.labName)) IS NOT NULL
    AND SAFE_CAST(l.labResult AS FLOAT64) IS NOT NULL
)
SELECT
  lab_name,
  STRING_AGG(DISTINCT unit_system, ' | ' ORDER BY unit_system LIMIT 10) AS unit_variants,
  COUNT(*) AS measurements,
  COUNT(DISTINCT patientUnitStayID) AS patient_stays,
  SAFE_DIVIDE(COUNT(DISTINCT patientUnitStayID),
              (SELECT COUNT(*) FROM main)) AS coverage,
  APPROX_QUANTILES(value, 100)[OFFSET(1)] AS p01,
  APPROX_QUANTILES(value, 100)[OFFSET(50)] AS median,
  APPROX_QUANTILES(value, 100)[OFFSET(99)] AS p99,
  MIN(value) AS observed_min,
  MAX(value) AS observed_max
FROM x
GROUP BY lab_name
HAVING patient_stays >= 25
ORDER BY patient_stays DESC, lab_name
LIMIT 300;
"""
result_04 = run_aggregate(
    "04_lab_name_coverage_audit.sql",
    SQL_04,
    "04_lab_name_coverage_audit.csv"
)

## 05_vitalperiodic_coverage_audit.sql

In [ ]:
SQL_05 = f"""
-- Shareable first-12-hour coverage and broad plausibility audit for periodic vitals.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
long AS (
  SELECT
    v.patientUnitStayID,
    z.feature,
    z.value,
    z.low_bound,
    z.high_bound
  FROM main m
  JOIN `{SOURCE_DATASET}.vitalperiodic` v USING (patientUnitStayID)
  CROSS JOIN UNNEST([
    STRUCT('temperature' AS feature, CAST(v.temperature AS FLOAT64) AS value, 25.0 AS low_bound, 45.0 AS high_bound),
    STRUCT('sao2' AS feature, CAST(v.sao2 AS FLOAT64) AS value, 40.0 AS low_bound, 100.0 AS high_bound),
    STRUCT('heart_rate' AS feature, CAST(v.heartRate AS FLOAT64) AS value, 20.0 AS low_bound, 250.0 AS high_bound),
    STRUCT('respiratory_rate' AS feature, CAST(v.respiration AS FLOAT64) AS value, 4.0 AS low_bound, 80.0 AS high_bound),
    STRUCT('cvp' AS feature, CAST(v.cvp AS FLOAT64) AS value, -10.0 AS low_bound, 50.0 AS high_bound),
    STRUCT('etco2' AS feature, CAST(v.etco2 AS FLOAT64) AS value, 5.0 AS low_bound, 100.0 AS high_bound),
    STRUCT('invasive_systolic_bp' AS feature, CAST(v.systemicSystolic AS FLOAT64) AS value, 40.0 AS low_bound, 300.0 AS high_bound),
    STRUCT('invasive_diastolic_bp' AS feature, CAST(v.systemicDiastolic AS FLOAT64) AS value, 20.0 AS low_bound, 200.0 AS high_bound),
    STRUCT('invasive_mean_bp' AS feature, CAST(v.systemicMean AS FLOAT64) AS value, 20.0 AS low_bound, 250.0 AS high_bound),
    STRUCT('pa_systolic' AS feature, CAST(v.paSystolic AS FLOAT64) AS value, 5.0 AS low_bound, 150.0 AS high_bound),
    STRUCT('pa_diastolic' AS feature, CAST(v.paDiastolic AS FLOAT64) AS value, 0.0 AS low_bound, 100.0 AS high_bound),
    STRUCT('pa_mean' AS feature, CAST(v.paMean AS FLOAT64) AS value, 0.0 AS low_bound, 120.0 AS high_bound),
    STRUCT('icp' AS feature, CAST(v.icp AS FLOAT64) AS value, -10.0 AS low_bound, 100.0 AS high_bound)
  ]) z
  WHERE v.observationOffset BETWEEN 0 AND 720
)
SELECT
  feature,
  COUNTIF(value IS NOT NULL) AS measurements,
  COUNT(DISTINCT IF(value IS NOT NULL, patientUnitStayID, NULL)) AS patient_stays,
  SAFE_DIVIDE(COUNT(DISTINCT IF(value IS NOT NULL, patientUnitStayID, NULL)),
              (SELECT COUNT(*) FROM main)) AS coverage,
  COUNTIF(value IS NOT NULL AND (value < low_bound OR value > high_bound)) AS outside_broad_qc_bounds,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(1)] AS plausible_p01,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(50)] AS plausible_median,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(99)] AS plausible_p99
FROM long
GROUP BY feature
ORDER BY patient_stays DESC, feature;
"""
result_05 = run_aggregate(
    "05_vitalperiodic_coverage_audit.sql",
    SQL_05,
    "05_vitalperiodic_coverage_audit.csv"
)

## 06_vitalaperiodic_coverage_audit.sql

In [ ]:
SQL_06 = f"""
-- Shareable first-12-hour coverage and broad plausibility audit for aperiodic hemodynamics.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
long AS (
  SELECT
    v.patientUnitStayID,
    z.feature,
    z.value,
    z.low_bound,
    z.high_bound
  FROM main m
  JOIN `{SOURCE_DATASET}.vitalaperiodic` v USING (patientUnitStayID)
  CROSS JOIN UNNEST([
    STRUCT('noninvasive_systolic_bp' AS feature, CAST(v.nonInvasiveSystolic AS FLOAT64) AS value, 40.0 AS low_bound, 300.0 AS high_bound),
    STRUCT('noninvasive_diastolic_bp' AS feature, CAST(v.nonInvasiveDiastolic AS FLOAT64) AS value, 20.0 AS low_bound, 200.0 AS high_bound),
    STRUCT('noninvasive_mean_bp' AS feature, CAST(v.nonInvasiveMean AS FLOAT64) AS value, 20.0 AS low_bound, 250.0 AS high_bound),
    STRUCT('paop' AS feature, CAST(v.paop AS FLOAT64) AS value, 0.0 AS low_bound, 60.0 AS high_bound),
    STRUCT('cardiac_output' AS feature, CAST(v.cardiacOutput AS FLOAT64) AS value, 0.5 AS low_bound, 20.0 AS high_bound),
    STRUCT('cardiac_index' AS feature, CAST(v.cardiacInput AS FLOAT64) AS value, 0.2 AS low_bound, 10.0 AS high_bound),
    STRUCT('svr' AS feature, CAST(v.svr AS FLOAT64) AS value, 100.0 AS low_bound, 4000.0 AS high_bound),
    STRUCT('svri' AS feature, CAST(v.svri AS FLOAT64) AS value, 100.0 AS low_bound, 7000.0 AS high_bound),
    STRUCT('pvr' AS feature, CAST(v.pvr AS FLOAT64) AS value, 0.0 AS low_bound, 1000.0 AS high_bound),
    STRUCT('pvri' AS feature, CAST(v.pvri AS FLOAT64) AS value, 0.0 AS low_bound, 2000.0 AS high_bound)
  ]) z
  WHERE v.observationOffset BETWEEN 0 AND 720
)
SELECT
  feature,
  COUNTIF(value IS NOT NULL) AS measurements,
  COUNT(DISTINCT IF(value IS NOT NULL, patientUnitStayID, NULL)) AS patient_stays,
  SAFE_DIVIDE(COUNT(DISTINCT IF(value IS NOT NULL, patientUnitStayID, NULL)),
              (SELECT COUNT(*) FROM main)) AS coverage,
  COUNTIF(value IS NOT NULL AND (value < low_bound OR value > high_bound)) AS outside_broad_qc_bounds,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(1)] AS plausible_p01,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(50)] AS plausible_median,
  APPROX_QUANTILES(IF(value BETWEEN low_bound AND high_bound, value, NULL),100)[OFFSET(99)] AS plausible_p99
FROM long
GROUP BY feature
ORDER BY patient_stays DESC, feature;
"""
result_06 = run_aggregate(
    "06_vitalaperiodic_coverage_audit.sql",
    SQL_06,
    "06_vitalaperiodic_coverage_audit.csv"
)

## 07_infusion_drug_inventory.sql

In [ ]:
SQL_07 = f"""
-- Shareable top first-12-hour infusion drug names in the locked main cohort.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
)
SELECT
  LOWER(TRIM(i.drugName)) AS drug_name,
  COUNT(*) AS records,
  COUNT(DISTINCT i.patientUnitStayID) AS patient_stays,
  SAFE_DIVIDE(COUNT(DISTINCT i.patientUnitStayID),
              (SELECT COUNT(*) FROM main)) AS coverage
FROM main m
JOIN `{SOURCE_DATASET}.infusiondrug` i USING (patientUnitStayID)
WHERE i.infusionOffset BETWEEN 0 AND 720
  AND NULLIF(TRIM(i.drugName),'') IS NOT NULL
GROUP BY drug_name
HAVING patient_stays >= 10
ORDER BY patient_stays DESC, drug_name
LIMIT 300;
"""
result_07 = run_aggregate(
    "07_infusion_drug_inventory.sql",
    SQL_07,
    "07_infusion_drug_inventory.csv"
)

## 08_respiratory_support_inventory.sql

In [ ]:
SQL_08 = f"""
-- Shareable first-12-hour respiratory-support coverage from respiratorycare and treatment.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
rc AS (
  SELECT
    'RESPIRATORYCARE' AS source,
    COALESCE(NULLIF(LOWER(TRIM(r.airwayType)),''),'[airway missing]') AS item,
    COUNT(*) AS records,
    COUNT(DISTINCT r.patientUnitStayID) AS patient_stays
  FROM main m
  JOIN `{SOURCE_DATASET}.respiratorycare` r USING (patientUnitStayID)
  WHERE COALESCE(r.respCareStatusOffset, r.ventStartOffset, 0) <= 720
    AND (r.ventStartOffset BETWEEN 0 AND 720 OR r.respCareStatusOffset BETWEEN 0 AND 720)
  GROUP BY item
),
tx AS (
  SELECT
    'TREATMENT' AS source,
    LOWER(TRIM(t.treatmentString)) AS item,
    COUNT(*) AS records,
    COUNT(DISTINCT t.patientUnitStayID) AS patient_stays
  FROM main m
  JOIN `{SOURCE_DATASET}.treatment` t USING (patientUnitStayID)
  WHERE t.treatmentOffset BETWEEN 0 AND 720
    AND REGEXP_CONTAINS(LOWER(t.treatmentString), r'ventil|intubat|airway|oxygen|cpap|bipap|high flow|mechanical ventilation')
  GROUP BY item
)
SELECT * FROM rc
UNION ALL
SELECT * FROM tx
ORDER BY source, patient_stays DESC, item
LIMIT 400;
"""
result_08 = run_aggregate(
    "08_respiratory_support_inventory.sql",
    SQL_08,
    "08_respiratory_support_inventory.csv"
)

## 09_past_history_inventory.sql

In [ ]:
SQL_09 = f"""
-- Shareable top pre-existing history terms documented by the 12-hour landmark.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
)
SELECT
  LOWER(TRIM(p.pastHistoryPath)) AS history_path,
  LOWER(TRIM(COALESCE(p.pastHistoryValueText, p.pastHistoryValue))) AS history_value,
  COUNT(*) AS records,
  COUNT(DISTINCT p.patientUnitStayID) AS patient_stays,
  SAFE_DIVIDE(COUNT(DISTINCT p.patientUnitStayID),
              (SELECT COUNT(*) FROM main)) AS coverage
FROM main m
JOIN `{SOURCE_DATASET}.pasthistory` p USING (patientUnitStayID)
WHERE p.pastHistoryOffset <= 720
  AND NULLIF(TRIM(p.pastHistoryPath),'') IS NOT NULL
GROUP BY history_path, history_value
HAVING patient_stays >= 20
ORDER BY patient_stays DESC, history_path, history_value
LIMIT 500;
"""
result_09 = run_aggregate(
    "09_past_history_inventory.sql",
    SQL_09,
    "09_past_history_inventory.csv"
)

## 10_landmark_leakage_audit.sql

In [ ]:
SQL_10 = f"""
-- Shareable audit proving source records selected for candidate predictors are within 0–720 min.
WITH main AS (
  SELECT patientUnitStayID
  FROM `{TARGET_DATASET}.cohort_outcome_v1`
  WHERE eligible_main_cohort = 1
),
checks AS (
  SELECT 'lab' AS source,
         COUNTIF(l.labResultOffset BETWEEN 0 AND 720) AS records_in_window,
         COUNTIF(l.labResultOffset > 720) AS records_after_landmark_in_source,
         MIN(IF(l.labResultOffset BETWEEN 0 AND 720,l.labResultOffset,NULL)) AS min_selected_offset,
         MAX(IF(l.labResultOffset BETWEEN 0 AND 720,l.labResultOffset,NULL)) AS max_selected_offset
  FROM main m JOIN `{SOURCE_DATASET}.lab` l USING(patientUnitStayID)
  UNION ALL
  SELECT 'vitalperiodic',
         COUNTIF(v.observationOffset BETWEEN 0 AND 720),
         COUNTIF(v.observationOffset > 720),
         MIN(IF(v.observationOffset BETWEEN 0 AND 720,v.observationOffset,NULL)),
         MAX(IF(v.observationOffset BETWEEN 0 AND 720,v.observationOffset,NULL))
  FROM main m JOIN `{SOURCE_DATASET}.vitalperiodic` v USING(patientUnitStayID)
  UNION ALL
  SELECT 'vitalaperiodic',
         COUNTIF(v.observationOffset BETWEEN 0 AND 720),
         COUNTIF(v.observationOffset > 720),
         MIN(IF(v.observationOffset BETWEEN 0 AND 720,v.observationOffset,NULL)),
         MAX(IF(v.observationOffset BETWEEN 0 AND 720,v.observationOffset,NULL))
  FROM main m JOIN `{SOURCE_DATASET}.vitalaperiodic` v USING(patientUnitStayID)
  UNION ALL
  SELECT 'infusiondrug',
         COUNTIF(i.infusionOffset BETWEEN 0 AND 720),
         COUNTIF(i.infusionOffset > 720),
         MIN(IF(i.infusionOffset BETWEEN 0 AND 720,i.infusionOffset,NULL)),
         MAX(IF(i.infusionOffset BETWEEN 0 AND 720,i.infusionOffset,NULL))
  FROM main m JOIN `{SOURCE_DATASET}.infusiondrug` i USING(patientUnitStayID)
  UNION ALL
  SELECT 'pasthistory',
         COUNTIF(p.pastHistoryOffset <= 720),
         COUNTIF(p.pastHistoryOffset > 720),
         MIN(IF(p.pastHistoryOffset <= 720,p.pastHistoryOffset,NULL)),
         MAX(IF(p.pastHistoryOffset <= 720,p.pastHistoryOffset,NULL))
  FROM main m JOIN `{SOURCE_DATASET}.pasthistory` p USING(patientUnitStayID)
)
SELECT * FROM checks ORDER BY source;
"""
result_10 = run_aggregate(
    "10_landmark_leakage_audit.sql",
    SQL_10,
    "10_landmark_leakage_audit.csv"
)

## Güvenli paylaşım paketi

In [ ]:
share_files = [
    '01_cohort_integrity_audit.csv',
    '02_source_column_inventory.csv',
    '03_demographic_coverage_audit.csv',
    '04_lab_name_coverage_audit.csv',
    '05_vitalperiodic_coverage_audit.csv',
    '06_vitalaperiodic_coverage_audit.csv',
    '07_infusion_drug_inventory.csv',
    '08_respiratory_support_inventory.csv',
    '09_past_history_inventory.csv',
    '10_landmark_leakage_audit.csv'
]
manifest = []
for filename in share_files:
    path = os.path.join(DRIVE_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing expected aggregate output: {path}")
    data = open(path, 'rb').read()
    manifest.append({"filename": filename, "size_bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()})

manifest_path = os.path.join(DRIVE_OUTPUT_DIR, 'MANIFEST_FEATURE_INVENTORY.json')
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

zip_path = os.path.join(DRIVE_OUTPUT_DIR, 'AKI_V2_FEATURE_INVENTORY_AGGREGATES.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for filename in share_files:
        z.write(os.path.join(DRIVE_OUTPUT_DIR, filename), arcname=filename)
    z.write(manifest_path, arcname='MANIFEST_FEATURE_INVENTORY.json')

print('Shareable aggregate package:', zip_path)
print('Do not export or upload patient-level tables from BigQuery dataset:', TARGET_DATASET)